# Parte 1 — Análise Exploratória de Dados (EDA)

Objetivo: entender os dados antes de modelar e responder às perguntas da Parte 1 do desafio.
As justificativas de cada decisão estão no [README](../README.md#parte-1--análise-exploratória).

Roteiro: 1. Contexto 
· 2. Divisão treino/teste 
· 3. Formato e tipos 
· 4. Integridade 
· 5. Distribuição de cada variável 
· 6. Relação entre pares de variáveis 
· 7. Efeito de uma terceira variável 
· 8. Registro dos achados

## Passo 1 — Contexto

- Linha: uma campanha (um vídeo veiculado em uma plataforma).
- Alvo: `klike_score` (0 a 100), qualidade geral do criativo segundo a IA da Klike.
- Esperado: 500 registros e 26 colunas.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/klike_challenge_dataset.csv")
df.shape

(500, 26)

## Passo 2 — Divisão treino/teste

Separamos o teste antes de explorar, para que nenhuma decisão seja influenciada por ele (evitar *data snooping*).
Daqui em diante, toda a análise usa só o `train`.

- 80% treino (400) / 20% teste (100)
- `random_state=42`: divisão reprodutível
- `stratify=platform`: mesmas proporções de plataforma nos dois conjuntos

In [2]:
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["platform"])
len(train), len(test)

(400, 100)

In [3]:
proporcoes = pd.DataFrame({
    "base": df["platform"].value_counts(normalize=True),
    "treino": train["platform"].value_counts(normalize=True),
    "teste": test["platform"].value_counts(normalize=True),
})
(proporcoes * 100).round(1)

,base,treino,teste
platform,,,
Meta,48.2,48.2,48.0
TikTok,35.2,35.2,35.0
LinkedIn,16.6,16.5,17.0


Proporções preservadas: a maior diferença em relação à base é de 0,4 ponto percentual (LinkedIn).

## Passo 3 — Formato e tipos

Conferir se cada coluna foi lida com o tipo certo e quais categorias existem. Só observamos aqui; as conversões de tipo ficam para o pré-processamento.

In [4]:
exemplos = train.head().T
exemplos.columns = [f"Exemplo {i}" for i in range(1, 6)]
exemplos

,Exemplo 1,Exemplo 2,Exemplo 3,Exemplo 4,Exemplo 5
campaign_id,KLK-0057,KLK-0301,KLK-0088,KLK-0230,KLK-0485
date,2024-02-07,2024-02-10,2024-10-24,2024-05-05,2024-01-05
platform,Meta,Meta,TikTok,TikTok,TikTok
category,E-commerce,Branding,E-commerce,E-commerce,Branding
objective,conversions,awareness,traffic,traffic,engagement
target_audience_age,18-24,35-44,18-24,18-24,18-24
is_retargeting,False,False,False,False,True
video_duration_s,13.0,11.0,11.0,5.0,11.0
format,quadrado,vertical,vertical,vertical,vertical
has_subtitle,True,True,True,False,False


In [5]:
tipos = pd.DataFrame({
    "tipo": train.dtypes.astype(str),
    "preenchidos": train.notna().sum(),
    "faltantes": train.isna().sum(),
    "exemplo": train.apply(lambda col: col.dropna().iloc[0]),
})
tipos

,tipo,preenchidos,faltantes,exemplo
campaign_id,object,400,0,KLK-0057
date,object,400,0,2024-02-07
platform,object,400,0,Meta
category,object,400,0,E-commerce
objective,object,400,0,conversions
target_audience_age,object,400,0,18-24
is_retargeting,bool,400,0,False
video_duration_s,float64,370,30,13.0
format,object,400,0,quadrado
has_subtitle,object,364,36,True


Verificação das categorias contra o dicionário de dados do desafio. Uma categoria nova não é tratada como erro, apenas sinalizada como "fora do dicionário" (ver README, Passo 3).

In [6]:
# Categorias esperadas segundo o dicionário de dados do desafio
CATEGORIAS_ESPERADAS = {
    "platform": ["Meta", "TikTok", "LinkedIn"],
    "category": ["E-commerce", "SaaS", "App Install", "Branding", "Lead Gen"],
    "objective": ["awareness", "traffic", "conversions", "engagement", "app_install"],
    "target_audience_age": ["18-24", "25-34", "35-44", "45+"],
    "format": ["vertical", "horizontal", "quadrado"],
    "text_density": ["low", "medium", "high"],
}


def checar_categorias(col):
    encontradas = set(train[col].dropna().unique())
    esperadas = set(CATEGORIAS_ESPERADAS[col])
    problemas = []
    if encontradas - esperadas:
        problemas.append("fora do dicionário: " + ", ".join(sorted(encontradas - esperadas)))
    if esperadas - encontradas:
        problemas.append("ausentes: " + ", ".join(sorted(esperadas - encontradas)))
    return "; ".join(problemas) or "ok"


pd.DataFrame(
    {"resultado": [checar_categorias(col) for col in CATEGORIAS_ESPERADAS]},
    index=pd.Index(list(CATEGORIAS_ESPERADAS), name="coluna"),
)

,resultado
coluna,
platform,ok
category,ok
objective,ok
target_audience_age,ok
format,ok
text_density,ok


Observações:
- Tipos a corrigir no pré-processamento: `date` (texto → data), `has_subtitle` (texto por causa de 36 vazios → verdadeiro/falso), `text_density` e `target_audience_age` (categorias com ordem natural, hoje sem ordem).
- Categorias consistentes com o dicionário de dados, sem variações de escrita.
- 7 colunas com valores faltantes (20 a 36 cada), analisadas no Passo 4.

## Passo 4 — Integridade

### 4.1 Valores faltantes

Quantos valores faltam, onde, e se faltam ao acaso ou ligados a outras características da campanha.

In [7]:
colunas_com_faltantes = train.columns[train.isna().any()].tolist()

faltantes = pd.DataFrame({
    "faltantes": train[colunas_com_faltantes].isna().sum(),
    "% do total": train[colunas_com_faltantes].isna().mean() * 100,
})
faltantes.style.format({"% do total": "{:.1f}%"})

,faltantes,% do total
video_duration_s,30,7.5%
has_subtitle,36,9.0%
music_voice_ratio,29,7.2%
cpc,20,5.0%
revenue,22,5.5%
avg_watch_time_s,22,5.5%
engagement_rate,27,6.8%


Entre 5% e 9% de vazios por coluna: pouco o bastante para nenhuma coluna precisar ser descartada.

In [8]:
vazios_por_campanha = train.isna().sum(axis=1).value_counts().sort_index()

tabela_vazios = pd.DataFrame(
    {
        "campanhas": vazios_por_campanha.values,
        "% do total": vazios_por_campanha.values / len(train) * 100,
    },
    index=pd.Index(["nenhum" if n == 0 else str(n) for n in vazios_por_campanha.index],
                   name="campos vazios na campanha"),
)
tabela_vazios.style.format({"% do total": "{:.1f}%"})

,campanhas,% do total
campos vazios na campanha,,
nenhum,250,62.5%
1,118,29.5%
2,28,7.0%
3,4,1.0%


62,5% das campanhas estão completas e quase todas as demais têm um único campo vazio. Os vazios estão espalhados, e não concentrados em poucas campanhas problemáticas.

In [9]:
# % de faltantes de cada coluna dentro de cada plataforma
(train[colunas_com_faltantes].isna().groupby(train["platform"]).mean() * 100).style.format("{:.1f}%")

,video_duration_s,has_subtitle,music_voice_ratio,cpc,revenue,avg_watch_time_s,engagement_rate
platform,,,,,,,
LinkedIn,9.1%,9.1%,12.1%,3.0%,6.1%,1.5%,7.6%
Meta,5.7%,8.3%,6.2%,5.2%,4.7%,5.2%,6.7%
TikTok,9.2%,9.9%,6.4%,5.7%,6.4%,7.8%,6.4%


Taxas parecidas entre plataformas. A maior (`music_voice_ratio` no LinkedIn, 12,1%) corresponde a apenas 8 campanhas.

Teste: para cada coluna com faltantes, comparamos as campanhas com e sem o valor vazio nas demais variáveis.
Categóricas: teste qui-quadrado. Numéricas: teste de Mann-Whitney. Listamos as diferenças com p < 0,05.

In [10]:
from scipy.stats import chi2_contingency, mannwhitneyu

VARIAVEIS_CATEGORICAS = ["platform", "category", "objective", "target_audience_age", "format",
                         "text_density", "is_retargeting", "has_hook", "has_face", "has_cta"]
VARIAVEIS_NUMERICAS = ["klike_score", "impressions", "spend", "ctr", "video_duration_s",
                       "engagement_rate", "roas"]


def comparar_com_e_sem_faltante(col):
    vazio = train[col].isna()
    resultados = []
    for var in VARIAVEIS_CATEGORICAS:
        if var != col:
            p = chi2_contingency(pd.crosstab(train[var], vazio))[1]
            resultados.append({"variável": var, "p": p, "mediana com vazio": None, "mediana sem vazio": None})
    for var in VARIAVEIS_NUMERICAS:
        if var != col:
            com, sem = train.loc[vazio, var].dropna(), train.loc[~vazio, var].dropna()
            resultados.append({"variável": var, "p": mannwhitneyu(com, sem).pvalue,
                               "mediana com vazio": com.median(), "mediana sem vazio": sem.median()})
    return pd.DataFrame(resultados).assign(coluna=col)


testes = pd.concat([comparar_com_e_sem_faltante(col) for col in colunas_com_faltantes], ignore_index=True)
print(f"Total de comparações: {len(testes)}")
testes.loc[testes["p"] < 0.05, ["coluna", "variável", "p", "mediana com vazio", "mediana sem vazio"]].round(3)

Total de comparações: 117


,coluna,variável,p,mediana com vazio,mediana sem vazio
62,cpc,spend,0.048,2387.335,966.255
63,cpc,ctr,0.042,0.101,0.063
95,avg_watch_time_s,impressions,0.002,9045.000,25331.000
96,avg_watch_time_s,spend,0.003,354.015,1107.300


Foram 117 comparações. Com o limite de p < 0,05, cerca de 6 dariam diferença só por acaso, e encontramos 4: o resultado geral é compatível com vazios ao acaso (MCAR). A exceção que merece atenção é `avg_watch_time_s`, com p bem menores e medianas de impressões e gasto cerca de três vezes menores nas campanhas com vazio. Olhamos esse caso mais de perto abaixo.

In [11]:
# % de campanhas sem avg_watch_time_s em cada faixa de impressões (5 faixas com 80 campanhas cada)
faixa_impressoes = pd.qcut(train["impressions"], 5,
                           labels=["1 (menores)", "2", "3", "4", "5 (maiores)"])
vazio_watch = train["avg_watch_time_s"].isna()

pd.DataFrame({
    "campanhas sem o valor": vazio_watch.groupby(faixa_impressoes, observed=True).sum(),
    "% da faixa": vazio_watch.groupby(faixa_impressoes, observed=True).mean() * 100,
}).rename_axis("faixa de impressões").style.format({"% da faixa": "{:.1f}%"})

,campanhas sem o valor,% da faixa
faixa de impressões,,
1 (menores),11,13.8%
2,3,3.8%
3,3,3.8%
4,4,5.0%
5 (maiores),1,1.2%


Metade dos vazios de `avg_watch_time_s` (11 de 22) está nos 20% de campanhas com menos impressões. Uma explicação plausível: com poucas visualizações, a plataforma não reporta o tempo médio assistido.

Observações do 4.1:
- Os vazios de 6 das 7 colunas são compatíveis com faltantes ao acaso (MCAR).
- `avg_watch_time_s` falta mais em campanhas pequenas (poucas impressões): vazios ligados a outra coluna (MAR). O tratamento precisa levar isso em conta.
- O que fazer com os vazios fica para o pré-processamento (ver README, Passo 4.1).
- `cpc` e `revenue` talvez possam ser recalculados a partir de outras colunas (`spend / clicks` e `roas × spend`); isso é verificado no 4.3.